# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# `metadata` is an mlcroissant.Metadata object, displaying key fields for context
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n\nPublished: {metadata.date_published}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets and their field and column `@id`s.

In [ ]:
# Display all RecordSets present in the dataset

print("Available Record Sets (@id and name):\n")
record_sets = []
for record_set in metadata.record_sets:
    print(f"  @id: {record_set.id}")
    print(f"    name: {record_set.name}")
    record_sets.append(record_set.id)
    # Show associated fields for each record set
    field_ids = [f.id for f in record_set.fields]
    print(f"    Fields: {field_ids}")
    if hasattr(record_set, 'columns') and record_set.columns:
        column_ids = [c.id for c in record_set.columns]
        print(f"    Columns: {column_ids}")
    print()
# Pick the first record set for later demonstration
selected_record_set = record_sets[0] if record_sets else None

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s identified in the previous section.

In [ ]:
# Extract data for each record set and show the columns

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nRecord set '@id': {record_set_id}, Shape: {df.shape}")
    print("  Columns:", df.columns.tolist())

# Example: Show first few rows for the selected record set
if selected_record_set:
    display_cols = dataframes[selected_record_set].columns.tolist()
    print(f"\nPreview for Record Set {selected_record_set}:")
    display(dataframes[selected_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section shows examples: removing outliers, normalizing a numeric field, and grouping by a categorical variable.

> **Note:** Change `numeric_field_id` and `group_field_id` to match column names (which are usually the field `@id`s) observed above.

In [ ]:
# --- Replace with IDs from your dataset ---
# For demonstration, pick the first record set and the first numeric and group fields.

import numpy as np

if selected_record_set:
    df = dataframes[selected_record_set].copy()
    numeric_field_id = None
    group_field_id = None
    # Try to detect a numeric field
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    # Try to detect a group/categorical field
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            group_field_id = col
            break
    print(f"Numeric field selected: {numeric_field_id}")
    print(f"Grouping field selected: {group_field_id}\n")

    # Filtering: keep rows where numeric > threshold
    threshold = df[numeric_field_id].quantile(0.75) if numeric_field_id else None
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group-by
        if group_field_id:
            print(f"\nGrouped mean by '{group_field_id}':")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            display(grouped.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the distribution of the selected numeric field and show a boxplot by group (if a grouping field is present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to explore, filter, and visualize the FAIR^2 protein abundance dataset using the `mlcroissant` API. With the provided record set and field `@id`s, you can further process and analyze the data for your own research.

Feel free to customize the analysis or adapt these steps to other datasets with Croissant schemas!